# Biblical Entity Force Graph
Builds an interactive force graph from raindrop.io Wikipedia bookmarks,
using the Wikipedia PHP API to discover internal link relationships.

In [ ]:
import json
import time
import re
from pathlib import Path
from urllib.parse import urlparse, unquote

import polars as pl
import requests
from cosmograph import cosmo

## 1. Load Raindrop Export

In [ ]:
df_raw = pl.read_csv("raindrop-export-2026-04-02/export.csv")

def extract_wiki_title(url: str) -> str | None:
    """Extract clean article title from a Wikipedia URL."""
    try:
        parsed = urlparse(url)
        if "wikipedia.org" not in parsed.netloc:
            return None
        path = parsed.path
        match = re.match(r"/wiki/(.+)", path)
        if not match:
            return None
        title = unquote(match.group(1)).replace("_", " ").split("#")[0].strip()
        return title if title else None
    except Exception:
        return None

wiki_titles = df_raw["url"].map_elements(extract_wiki_title, return_dtype=pl.String)
df_raw = df_raw.with_columns(wiki_titles.alias("wiki_title"))
wiki_articles = (
    df_raw.filter(pl.col("wiki_title").is_not_null())
    .select(["wiki_title", "title", "url"])
    .unique("wiki_title")
)

print(f"Total bookmarks: {len(df_raw)}")
print(f"Wikipedia articles: {len(wiki_articles)}")
wiki_articles.head(10)

## 2. Fetch Internal Links via Wikipedia PHP API
For each article, fetch which other articles in our set it links to.

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
CACHE_FILE = Path(".wiki_links_cache.json")

cache: dict = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

def fetch_links(title: str) -> list[str]:
    """Fetch all internal links from a Wikipedia article (cached)."""
    if title in cache:
        return cache[title]

    links = []
    params = {
        "action": "query",
        "prop": "links",
        "titles": title,
        "pllimit": "max",
        "format": "json",
        "plnamespace": 0,
    }
    while True:
        r = requests.get(WIKI_API, params=params, headers={"User-Agent": "wiki-cosmo-graph/1.0"})
        data = r.json()
        pages = data.get("query", {}).get("pages", {})
        for page in pages.values():
            for link in page.get("links", []):
                links.append(link["title"])
        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.1)

    cache[title] = links
    CACHE_FILE.write_text(json.dumps(cache, indent=2))
    return links

article_titles = set(wiki_articles["wiki_title"].to_list())
missing = [t for t in article_titles if t not in cache]
if missing:
    print(f"Fetching {len(missing)} missing articles...")
    for i, title in enumerate(missing):
        fetch_links(title)
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(missing)}")

print(f"Cache: {len(cache)} articles | Missing fetched: {len(missing)}")

## 3. Build Nodes & Edges

In [ ]:
# Nodes: one per Wikipedia article, with URL for reference
nodes_df = wiki_articles.rename({"wiki_title": "id"}).with_columns(
    pl.lit("seed").alias("node_type"),
    (
        "https://en.wikipedia.org/wiki/"
        + pl.col("id").str.replace_all(" ", "_")
    ).alias("wiki_url"),
)

# Edges: A→B if B's title appears in A's internal link list (bidirectional check)
edges = []
for src in article_titles:
    for linked in cache.get(src, []):
        if linked in article_titles and linked != src:
            edges.append({"source": src, "target": linked})

edges_df = pl.DataFrame(edges).unique(["source", "target"]) if edges else pl.DataFrame({"source": [], "target": []})

print(f"Nodes: {len(nodes_df)}")
print(f"Edges: {len(edges_df)}")
edges_df.head()

## 4. Render Force Graph

In [ ]:
graph = cosmo(
    points=nodes_df.to_pandas(),
    links=edges_df.to_pandas(),
    point_id_by="id",
    point_label_by="id",
    point_color_by="node_type",
    point_color_by_map={"seed": "#60a5fa"},
    link_source_by="source",
    link_target_by="target",
    link_width=1.5,
    link_color="#94a3b8",
    link_greyout_opacity=0.05,
    point_greyout_opacity=0.1,
    select_point_on_click=True,
    render_hovered_point_ring=True,
    link_arrows=False,
    simulation_gravity=0.1,
    simulation_repulsion=2.0,
    simulation_link_spring=1.0,
    simulation_friction=0.85,
    show_labels=True,
    show_hovered_point_label=True,
    fit_view_on_init=True,
    background_color="#0f172a",
    point_label_color="#e2e8f0",
)
graph